# 📊 Sector & Peer Comparison

Compare valuation multiples, growth rates, and profitability across peer companies.

**Usage:** Enter tickers, run all cells to see comparison tables and charts.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.data_engine import get_peer_metrics, get_profile

plt.style.use('ggplot')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 100

print('✅ Ready.')

## 1. Select Peer Group

In [ ]:
# Semiconductor / AI peers
TICKERS = ['NVDA', 'AVGO', 'AMD', 'INTC', 'QCOM', 'MRVL', 'TXN', 'MU']

# Or try enterprise software:
# TICKERS = ['ORCL', 'MSFT', 'CRM', 'ADBE', 'SAP', 'IBM', 'NOW']

print(f'Comparing {len(TICKERS)} companies: {", ".join(TICKERS)}')

## 2. Valuation Multiples Table

In [ ]:
df = get_peer_metrics(TICKERS)

if not df.empty:
    # Select and rename columns for display
    display_cols = {
        'ticker': 'Ticker',
        'name': 'Company',
        'sector': 'Sector',
        'market_cap': 'Market Cap',
        'pe_ratio': 'P/E',
        'pb_ratio': 'P/B',
        'ev_to_ebitda': 'EV/EBITDA',
        'revenue_growth': 'Rev Growth',
        'gross_margin': 'Gross Margin',
        'operating_margin': 'Op Margin',
        'net_margin': 'Net Margin',
        'roe': 'ROE',
    }
    available = {k: v for k, v in display_cols.items() if k in df.columns}
    display_df = df[list(available.keys())].rename(columns=available)
    
    # Format
    for col in display_df.columns:
        if col == 'Ticker':
            continue
        if col == 'Market Cap':
            display_df[col] = display_df[col].apply(
                lambda x: f'${x/1e12:.2f}T' if x > 1e12 else f'${x/1e9:.0f}B' if pd.notna(x) else '—')
        elif col == 'Company' or col == 'Sector':
            continue
        else:
            display_df[col] = display_df[col].apply(
                lambda x: f'{x:.1f}%' if 'margin' in col.lower() or 'roe' in col.lower() or 'growth' in col.lower()
                else f'{x:.1f}x' if pd.notna(x) else '—')
    
    display(display_df)
    print(f'\n✅ {len(df)} companies compared.')
    print(f'   Median P/E: {df["pe_ratio"].median():.1f}x')
    print(f'   Median EV/EBITDA: {df["ev_to_ebitda"].median():.1f}x' if 'ev_to_ebitda' in df.columns else '')
    print(f'   Median Rev Growth: {df["revenue_growth"].median():.1%}' if 'revenue_growth' in df.columns else '')
else:
    print('⚠️  No peer data available.')

## 3. Growth vs Valuation Scatter

In [ ]:
if not df.empty and 'revenue_growth' in df.columns and 'pe_ratio' in df.columns:
    fig, ax = plt.subplots(figsize=(10, 7))
    
    valid = df[['ticker', 'revenue_growth', 'pe_ratio']].dropna()
    
    ax.scatter(valid['revenue_growth'] * 100, valid['pe_ratio'], s=150, alpha=0.7, c='steelblue')
    
    for _, row in valid.iterrows():
        ax.annotate(row['ticker'], 
                    (row['revenue_growth'] * 100, row['pe_ratio']),
                    xytext=(7, 5), textcoords='offset points', fontsize=10, fontweight='bold')
    
    ax.set_xlabel('Revenue Growth (%)')
    ax.set_ylabel('P/E Ratio')
    ax.set_title('Growth vs Valuation — Peer Comparison')
    ax.axhline(y=valid['pe_ratio'].median(), color='red', linestyle='--', alpha=0.5, label=f'Median P/E: {valid["pe_ratio"].median():.1f}x')
    ax.axvline(x=valid['revenue_growth'].median() * 100, color='gray', linestyle='--', alpha=0.5)
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  Insufficient data for scatter plot.')

## 4. Profitability Comparison (Bar Chart)

In [ ]:
margin_cols = ['gross_margin', 'operating_margin', 'net_margin']
available_margins = [c for c in margin_cols if c in df.columns]

if available_margins and not df.empty:
    fig, ax = plt.subplots(figsize=(10, 6))
    
    x = np.arange(len(df))
    width = 0.25
    colors = ['steelblue', 'orange', 'green']
    
    for i, (col, color) in enumerate(zip(available_margins, colors[:len(available_margins)])):
        values = df[col].values * 100 if len(df) > 0 else []
        ax.bar(x + i * width, values, width, label=col.replace('_', ' ').title(), color=color, alpha=0.8)
    
    ax.set_xticks(x + width * (len(available_margins) - 1) / 2)
    ax.set_xticklabels(df['ticker'].values, fontweight='bold')
    ax.set_ylabel('%')
    ax.set_title('Margin Comparison')
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print('⚠️  No margin data.')

## 5. Identify Outliers

In [ ]:
if not df.empty:
    print('### Potentially Undervalued (Low P/E + High Growth):')
    if 'pe_ratio' in df.columns and 'revenue_growth' in df.columns:
        median_pe = df['pe_ratio'].median()
        median_growth = df['revenue_growth'].median()
        value = df[(df['pe_ratio'] < median_pe) & (df['revenue_growth'] > median_growth)]
        for _, row in value.iterrows():
            print(f"  ✅ {row['ticker']}: P/E={row['pe_ratio']:.1f}x, Growth={row['revenue_growth']:.1%}")
        if value.empty:
            print('  (none found)')
    
    print()
    print('### Potentially Overvalued (High P/E + Low Growth):')
    if 'pe_ratio' in df.columns and 'revenue_growth' in df.columns:
        overvalued = df[(df['pe_ratio'] > median_pe) & (df['revenue_growth'] < median_growth)]
        for _, row in overvalued.iterrows():
            print(f"  ⚠️  {row['ticker']}: P/E={row['pe_ratio']:.1f}x, Growth={row['revenue_growth']:.1%}")
        if overvalued.empty:
            print('  (none found)')
else:
    print('⚠️  No data.')

---
*Generated by AI Investment System*